In [1]:
!pip install tensorflow kaggle

In [4]:
!wget https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/6aeb4f15-9a34-4d88-8d27-6fddfae0b1e3/file_downloaded

--2026-03-10 15:36:10--  https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/6aeb4f15-9a34-4d88-8d27-6fddfae0b1e3/file_downloaded
Resolving data.mendeley.com (data.mendeley.com)... 162.159.130.86, 162.159.133.86
Connecting to data.mendeley.com (data.mendeley.com)|162.159.130.86|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1174 (1.1K) [application/json]
Saving to: ‘file_downloaded.1’

file_downloaded.1   100%[===================>]   1.15K  --.-KB/s    in 0s      

2026-03-10 15:36:11 (18.1 MB/s) - ‘file_downloaded.1’ saved [1174/1174]



In [6]:
!wget https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz
!tar -xzf flower_photos.tgz

--2026-03-10 15:37:13--  https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.212.207, 173.194.215.207, 173.194.216.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.212.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228813984 (218M) [application/x-compressed-tar]
Saving to: ‘flower_photos.tgz’

flower_photos.tgz   100%[===================>] 218.21M   154MB/s    in 1.4s    

2026-03-10 15:37:14 (154 MB/s) - ‘flower_photos.tgz’ saved [228813984/228813984]



In [7]:
!tar -xzf flower_photos.tgz

In [9]:
!ls flower_photos

daisy  dandelion  LICENSE.txt  roses  sunflowers  tulips


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

dataset = "flower_photos"

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train = datagen.flow_from_directory(
    dataset,
    target_size=(224,224),
    batch_size=32,
    subset='training'
)

val = datagen.flow_from_directory(
    dataset,
    target_size=(224,224),
    batch_size=32,
    subset='validation'
)

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_model.layers:
    layer.trainable=False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(train.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train, validation_data=val, epochs=3)

model.save("plant_disease_model.h5")

Found 2939 images belonging to 5 classes.
Found 731 images belonging to 5 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 186s 2s/step - accuracy: 0.7252 - loss: 0.7212 - val_accuracy: 0.8714 - val_loss: 0.4012
Epoch 2/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 167s 2s/step - accuracy: 0.9230 - loss: 0.2322 - val_accuracy: 0.8386 - val_loss: 0.4932
Epoch 3/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9436 - loss: 0.1657

In [11]:
from google.colab import files
files.download("plant_disease_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>